# data_quality_suite.ipynb v1.0

**Run order:** standalone — run anytime after the silver/gold materialisation notebooks. Not a dependency of any other notebook.

Runs the same known-bug regression checks as `tests/data_quality/run_checks.py` (loaded from `tests/data_quality/checks/*.sql` in this repo checkout) against `workspace.genealogy`, and writes results to `genealogy.data_quality_results` — the same Delta table the Python runner writes to, so trend data from both surfaces lives in one place.

**No Asana integration here.** CI (`run_checks.py` via GitHub Actions) is the sole owner of Asana task filing, so this notebook can be run freely — on a schedule or ad hoc — without racing CI to create duplicate tasks. This path is for native/scheduled runs and trend data, not alerting.

## Cell 1 — Locate the checks directory and load check files
## Cell 2 — Parse header + run each check, write to `data_quality_results`
## Cell 3 — Verification: pass/fail summary


## Cell 1 — Locate the checks directory

This notebook lives at the repo root; the check files live at `tests/data_quality/checks/` in the same Git-Repos checkout. Databricks Repos expose the checkout at `/Workspace` + the notebook's workspace path for direct file access (DBR 11.2+). If that mapping is wrong for this workspace, set the `checks_dir_override` widget instead of guessing further — the cell below fails loudly with the path it tried, rather than silently finding zero checks.


In [0]:
dbutils.widgets.text("checks_dir_override", "", "Checks dir override (optional)")
override = dbutils.widgets.get("checks_dir_override").strip()

import os

if override:
    CHECKS_DIR = override
else:
    notebook_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    repo_root = "/Workspace" + os.path.dirname(notebook_path)
    CHECKS_DIR = f"{repo_root}/tests/data_quality/checks"

if not os.path.isdir(CHECKS_DIR):
    raise FileNotFoundError(
        f"Checks directory not found at '{CHECKS_DIR}'. "
        "Set the checks_dir_override widget to the correct path for this workspace."
    )

check_files = sorted(
    os.path.join(CHECKS_DIR, f) for f in os.listdir(CHECKS_DIR) if f.endswith(".sql")
)
print(f"CHECKS_DIR = {CHECKS_DIR}")
print(f"Found {len(check_files)} check file(s):")
for f in check_files:
    print(" ", os.path.basename(f))


## Cell 2 — Parse header + run each check

Same tolerant line-prefix header parser as `run_checks.py` (kept in sync manually — there are only two small parsers, one per runtime, both reading the same `.sql` files, so the check *logic* itself is never duplicated). Each check's query is run via `spark.sql` rather than the `databricks-sql-connector` path the Python runner uses, since this notebook already has a live Spark session.


In [0]:
import json
import re
import uuid
from datetime import datetime, timezone
from decimal import Decimal

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, LongType, TimestampType
)

HEADER_KEYS = {
    "id", "title", "severity", "guards_bug", "known_failing",
    "existing_asana_task", "description",
}
SAMPLE_CAP = 20


def parse_check_file(path):
    with open(path) as fh:
        lines = fh.read().splitlines()
    header = {k: "" for k in HEADER_KEYS}
    current_key = None
    sql_start = len(lines)

    for i, line in enumerate(lines):
        if not line.strip():
            sql_start = i + 1
            break
        if not line.startswith("--"):
            sql_start = i
            break
        content = line[2:]
        if content.startswith(" "):
            content = content[1:]
        match = re.match(r"^(\w+):\s?(.*)$", content)
        if match and match.group(1) in HEADER_KEYS:
            current_key = match.group(1)
            value = match.group(2).strip()
            header[current_key] = "" if (current_key == "description" and value == ">") else value
        elif current_key == "description":
            header["description"] = (header["description"] + " " + content.strip()).strip()

    sql_text = "\n".join(lines[sql_start:]).strip()
    return {
        "id": header["id"].strip(),
        "title": header["title"].strip(),
        "severity": header["severity"].strip().lower(),
        "guards_bug": header["guards_bug"].strip() or None,
        "known_failing": header["known_failing"].strip().lower() == "true",
        "existing_asana_task": header["existing_asana_task"].strip() or None,
        "description": header["description"].strip(),
        "sql": sql_text,
    }


def json_default(value):
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, datetime):
        return value.isoformat()
    return str(value)


spark.sql("""
  CREATE TABLE IF NOT EXISTS genealogy.data_quality_results (
    run_id STRING,
    check_id STRING,
    run_at TIMESTAMP,
    status STRING,
    severity STRING,
    violation_count BIGINT,
    sample_violations STRING,
    known_failing BOOLEAN,
    guards_bug STRING,
    asana_task_gid STRING
  ) USING DELTA
""")

run_id = str(uuid.uuid4())
run_at = datetime.now(timezone.utc)
results = []
result_rows = []

for path in check_files:
    check = parse_check_file(path)
    df = spark.sql(check["sql"])
    rows = [r.asDict(recursive=True) for r in df.collect()]
    violation_count = len(rows)
    status = "pass" if violation_count == 0 else "fail"
    sample_violations = (
        json.dumps(rows[:SAMPLE_CAP], default=json_default) if status == "fail" else None
    )

    results.append({**check, "status": status, "violation_count": violation_count})
    result_rows.append(Row(
        run_id=run_id,
        check_id=check["id"],
        run_at=run_at,
        status=status,
        severity=check["severity"],
        violation_count=violation_count,
        sample_violations=sample_violations,
        known_failing=check["known_failing"],
        guards_bug=check["guards_bug"],
        asana_task_gid=None,  # this notebook never files Asana tasks -- see header note
    ))
    print(f"{check['id']:<8} {status:<6} {violation_count:>6} rows  ({check['severity']})")

schema = StructType([
    StructField("run_id", StringType()),
    StructField("check_id", StringType()),
    StructField("run_at", TimestampType()),
    StructField("status", StringType()),
    StructField("severity", StringType()),
    StructField("violation_count", LongType()),
    StructField("sample_violations", StringType()),
    StructField("known_failing", BooleanType()),
    StructField("guards_bug", StringType()),
    StructField("asana_task_gid", StringType()),
])
spark.createDataFrame(result_rows, schema=schema).write.mode("append").saveAsTable(
    "genealogy.data_quality_results"
)
print(f"\nrun_id: {run_id} — wrote {len(result_rows)} row(s) to genealogy.data_quality_results")


## Cell 3 — Verification


In [0]:
# Pass/fail summary for this run
passed = [r for r in results if r["status"] == "pass"]
failed = [r for r in results if r["status"] == "fail"]
newly_failed_critical = [
    r for r in failed if r["severity"] == "critical" and not r["known_failing"]
]

print(f"Checks run:            {len(results)}")
print(f"Passed:                {len(passed)}")
print(f"Failed:                {len(failed)}")
print(f"  of which known_failing (expected open bugs): {sum(1 for r in failed if r['known_failing'])}")
print(f"  of which NEW critical failures (expect 0):   {len(newly_failed_critical)}")

if newly_failed_critical:
    print("\nNEW critical failures — investigate and file/update an Asana task via the CI runner:")
    for r in newly_failed_critical:
        print(f"  {r['id']}: {r['title']} ({r['violation_count']} violating rows)")

spark.sql(f"""
  SELECT check_id, status, severity, violation_count, known_failing
  FROM genealogy.data_quality_results
  WHERE run_id = '{run_id}'
  ORDER BY check_id
""").display()
